# Lab 6 — Notebook 3: Multi-table queries and window functions

**What you'll do here:** write a 4-way join across `lineitem`, `orders`, `customer`, and `nation` to answer two business questions. Both would be painful in raw Hadoop MapReduce — slide 5 of the Week 07 deck lists this kind of join as exactly the case that motivated Spark.

**Business questions:**

1. **(Required)** For shipments in calendar year 1994, what is the total discounted revenue by customer nation?
2. **(Bonus)** Within each nation, who are the top 3 customers by 1994 revenue?


## Setup


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Lab6-N3-MultiTable")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider",
    )
    .config("spark.hadoop.fs.s3a.requester.pays.enabled", "true")
    .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-1.amazonaws.com")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

BASE = "s3a://tpch-torstengrabs-parquet/1GB/"
lineitem = spark.read.parquet(BASE + "lineitem/")
orders   = spark.read.parquet(BASE + "orders/")
customer = spark.read.parquet(BASE + "customer/")
nation   = spark.read.parquet(BASE + "nation/")

for name, df in [("lineitem", lineitem), ("orders", orders), ("customer", customer), ("nation", nation)]:
    print(f"{name:9s} — {len(df.columns)} columns")


## 4.1 Revenue per nation (20 pts)

Compute the total discounted revenue (`l_extendedprice * (1 - l_discount)`) by customer nation, for shipments where `l_shipdate` falls in calendar year 1994. Output one row per nation, sorted by revenue descending.

You'll need to join four tables in this order (or any equivalent):

- `lineitem` ⋈ `orders` on `l_orderkey = o_orderkey`
- ⋈ `customer` on `o_custkey = c_custkey`
- ⋈ `nation` on `c_nationkey = n_nationkey`

**Hints:**
- Use `df.join(other, on_column_or_condition, how="inner")` (deck slide 27).
- Apply the date filter as early in the chain as possible — Catalyst will push it down to the parquet scan anyway, but writing it explicitly close to `lineitem` makes the code's intent clearer.
- Use `F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount")))` for the revenue aggregate.


In [ ]:
# TODO: implement the revenue-by-nation query described above.
# revenue_by_nation = (lineitem
#     .filter(...)
#     .join(orders,   ...)
#     .join(customer, ...)
#     .join(nation,   ...)
#     .groupBy(...)
#     .agg(...)
#     .orderBy(...))
# revenue_by_nation.show(25, truncate=False)


**Expected output:** exactly 25 rows (TPC-H has 25 nations). Per-nation revenue should land in the **$1.28–1.38 billion** range — the top and bottom nations differ by only a few percent, because TPC-H is generated with uniform distributions across nations.


## 4.2 Read the 4-way join plan (10 pts)

Inspect Catalyst's plan and answer the three questions below.


In [ ]:
revenue_by_nation.explain(mode="extended")


**Questions** (replace each answer with yours):

1. **Predicate pushdown:** Where does the `l_shipdate` filter appear in the physical plan? Is it pushed down into the `Scan parquet` node for `lineitem`?
2. **Column pruning:** How many columns does the physical scan of `lineitem` report reading (look at the `ReadSchema` line on the `Scan parquet` node)? Compare to the 16 columns in the original table — what's pruned?
3. **Join algorithms:** What join algorithm did Catalyst pick for the join with `nation`? What about for the joins among `lineitem`, `orders`, and `customer`? (Look for `SortMergeJoin`, `BroadcastHashJoin`, or `ShuffleHashJoin` in the physical plan.) Why is the choice different — what does the size of `nation` (25 rows) vs. the others (150K to 6M rows) tell Catalyst?


**Answers** (replace with yours):

1. _TODO_
2. _TODO_
3. _TODO_


## 4.3 Top-3 customers per nation (10 pts — bonus)

Now extend 4.1 to identify, **within each nation**, the **three customers** with the highest 1994 revenue.

This is the canonical "top-N per group" pattern that requires a window function (deck slide 29). The idiom is:

1. Compute per-customer revenue (group by `n_name`, `c_custkey`, `c_name`).
2. Define a window partitioned by `n_name`, ordered by revenue descending.
3. Add a `row_number()` column.
4. Filter to `row_number <= 3` and sort.

Expected output: **75 rows** (25 nations × 3 customers each).


In [ ]:
# TODO: top-3 customers per nation, using a window function.
# Hint:
# customer_revenue = (lineitem
#     .filter(...)
#     .join(...).join(...).join(...)
#     .groupBy("n_name", "c_custkey", "c_name")
#     .agg(F.sum(...).alias("cust_revenue")))
#
# w = Window.partitionBy("n_name").orderBy(F.desc("cust_revenue"))
# top3 = (customer_revenue
#     .withColumn("rank_in_nation", F.row_number().over(w))
#     .filter(F.col("rank_in_nation") <= 3)
#     .orderBy("n_name", "rank_in_nation"))
# top3.show(75, truncate=False)


## 4.4 Plan inspection on the windowed query

Inspect the plan one more time and note the addition of the `Window` operator. The window introduces an **additional shuffle** (or at least an additional sort within the existing partitions) — find it in the plan.


In [ ]:
# TODO: call .explain(mode="formatted") on your top3 DataFrame


## 4.5 Spark UI screenshot

Take a screenshot of the **DAG visualization for your top-3 customers query** (from the Spark UI's Stages tab). Save it as `notebook3_screenshots/top3_dag.png`. Notice how many more stages exist than in Q1 (Notebook 2) — each `Exchange` boundary creates a new stage.


## Closing — stop the session


In [ ]:
spark.stop()
